Participant 2 Code

In [1]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import ks_2samp, mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = "participant_2_results"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Aesthetic config (consistent with project theme) ─────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#d0d7de",
    "axes.labelcolor":  "#24292f",
    "axes.titlecolor":  "#24292f",
    "xtick.color":      "#57606a",
    "ytick.color":      "#57606a",
    "text.color":       "#24292f",
    "grid.color":       "#f0f0f0",
    "grid.linestyle":   "--",
    "grid.linewidth":   0.6,
    "legend.facecolor": "#ffffff",
    "legend.edgecolor": "#d0d7de",
    "font.family":      "monospace",
    "figure.dpi":       130,
})

CRISIS_COLOR  = "#ff7b72"
NORMAL_COLOR  = "#3fb950"
ACCENT_COLOR  = "#58a6ff"
WARN_COLOR    = "#d29922"
NEUTRAL_COLOR = "#8b949e"
VOL_COLOR     = "#bc8cff"
WIN_COLORS    = {20: "#58a6ff", 60: "#bc8cff", 120: "#d29922"}

# ── Load preprocessed data from Participant 1 ─────────────────────────────────
print("=" * 70)
print("  PARTICIPANT 2 — ROLLING VOLATILITY COMPUTATION & COMPARATIVE ANALYSIS")
print("=" * 70)

df = pd.read_csv("FTSE100 Processed.csv", index_col=0)
df.index = pd.to_datetime(df.index, format='mixed', dayfirst=True)
df = df.dropna(subset=["Adaptive_Threshold"]).copy()
df = df.sort_index()

print(f"\n  Loaded: {len(df)} trading days  |  "
      f"Crisis: {df['Crisis_Indicator'].sum()}  |  "
      f"Non-Crisis: {(df['Crisis_Indicator']==0).sum()}")

ret = df["Log_Return"]
windows = [20, 60, 120]

# =============================================================================
# SECTION 1 — ROLLING VOLATILITY FEATURE COMPUTATION
# =============================================================================
print("\n" + "="*70)
print("  SECTION 1: COMPUTING 15-FEATURE ROLLING VOLATILITY MATRIX")
print("="*70)

features = {}

for w in windows:
    roll = ret.rolling(window=w, min_periods=w)

    # 1. Rolling Standard Deviation
    col = f"RSD_{w}"
    df[col] = roll.std()
    features[col] = {"family": "RSD", "window": w, "description": f"Rolling Std Dev ({w}d)"}

    # 2. Rolling Mean Absolute Deviation
    col = f"MAD_{w}"
    df[col] = roll.apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    features[col] = {"family": "MAD", "window": w, "description": f"Rolling MAD ({w}d)"}

    # 3. Rolling Variance
    col = f"VAR_{w}"
    df[col] = roll.var()
    features[col] = {"family": "VAR", "window": w, "description": f"Rolling Variance ({w}d)"}

    # 4. Rolling Range (max - min of returns in window)
    col = f"RNG_{w}"
    df[col] = roll.max() - roll.min()
    features[col] = {"family": "RNG", "window": w, "description": f"Rolling Range ({w}d)"}

    # 5. Rolling Kurtosis
    col = f"KURT_{w}"
    df[col] = roll.apply(lambda x: stats.kurtosis(x, fisher=True), raw=True)
    features[col] = {"family": "KURT", "window": w, "description": f"Rolling Kurtosis ({w}d)"}

feature_cols = list(features.keys())
print(f"\n  Computed {len(feature_cols)} features:")
for col, meta in features.items():
    non_null = df[col].notna().sum()
    print(f"    {col:<12}  |  {meta['description']:<30}  |  Non-null: {non_null}")

# =============================================================================
# SECTION 2 — DESCRIPTIVE STATISTICS BY REGIME
# =============================================================================
print("\n" + "="*70)
print("  SECTION 2: DESCRIPTIVE STATISTICS BY REGIME")
print("="*70)

crisis_df     = df[df["Crisis_Indicator"] == 1]
non_crisis_df = df[df["Crisis_Indicator"] == 0]

summary_rows = []
for col, meta in features.items():
    for regime_label, subset in [("Crisis", crisis_df), ("Non-Crisis", non_crisis_df), ("All", df)]:
        s = subset[col].dropna()
        if len(s) == 0:
            continue
        row = {
            "Feature":    col,
            "Family":     meta["family"],
            "Window":     meta["window"],
            "Regime":     regime_label,
            "N":          len(s),
            "Mean":       round(s.mean(), 8),
            "Median":     round(s.median(), 8),
            "Std":        round(s.std(), 8),
            "P10":        round(s.quantile(0.10), 8),
            "P90":        round(s.quantile(0.90), 8),
            "Min":        round(s.min(), 8),
            "Max":        round(s.max(), 8),
            "Skewness":   round(stats.skew(s), 4),
            "Kurtosis":   round(stats.kurtosis(s, fisher=True), 4),
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUT_DIR}/table01_descriptive_stats_by_regime.csv", index=False)

print(f"\n  TABLE 1 — Descriptive Statistics (Crisis vs Non-Crisis)")
print(f"  {'Feature':<12} {'Regime':<12} {'N':>6} {'Mean':>12} {'Median':>12} {'Std':>12} {'P10':>12} {'P90':>12}")
print(f"  {'-'*82}")
for _, row in summary_df[summary_df["Regime"] != "All"].iterrows():
    print(f"  {row['Feature']:<12} {row['Regime']:<12} {row['N']:>6} "
          f"{row['Mean']:>12.6f} {row['Median']:>12.6f} {row['Std']:>12.6f} "
          f"{row['P10']:>12.6f} {row['P90']:>12.6f}")

# =============================================================================
# SECTION 3 — NON-PARAMETRIC HYPOTHESIS TESTS
# =============================================================================
print("\n" + "="*70)
print("  SECTION 3: NON-PARAMETRIC HYPOTHESIS TESTS (KS + MANN-WHITNEY U)")
print("="*70)

test_rows = []
print(f"\n  TABLE 2 — KS Test & Mann-Whitney U: Crisis vs Non-Crisis")
print(f"  {'Feature':<12} {'KS Stat':>10} {'KS p':>12} {'MW Stat':>14} {'MW p':>12} {'Sig':>8}")
print(f"  {'-'*74}")

for col in feature_cols:
    c_vals  = crisis_df[col].dropna()
    nc_vals = non_crisis_df[col].dropna()
    if len(c_vals) < 10 or len(nc_vals) < 10:
        continue

    ks_stat, ks_p   = ks_2samp(c_vals, nc_vals)
    mw_stat, mw_p   = mannwhitneyu(c_vals, nc_vals, alternative="two-sided")

    if ks_p < 0.01 and mw_p < 0.01:
        sig = "***"
    elif ks_p < 0.05 and mw_p < 0.05:
        sig = "**"
    elif ks_p < 0.10 or mw_p < 0.10:
        sig = "*"
    else:
        sig = "ns"

    test_rows.append({
        "Feature": col, "Family": features[col]["family"], "Window": features[col]["window"],
        "KS_Stat": round(ks_stat, 4), "KS_p": round(ks_p, 8),
        "MW_Stat": round(mw_stat, 4), "MW_p": round(mw_p, 8),
        "Significance": sig,
    })
    print(f"  {col:<12} {ks_stat:>10.4f} {ks_p:>12.6f} {mw_stat:>14.2f} {mw_p:>12.6f} {sig:>8}")

test_df = pd.DataFrame(test_rows)
test_df.to_csv(f"{OUT_DIR}/table02_hypothesis_tests.csv", index=False)

# Effect size: Cohen's d for each feature
print(f"\n  TABLE 3 — Effect Size (Cohen's d) per Feature")
print(f"  {'Feature':<12} {'Cohen_d':>10} {'Magnitude':>12}")
print(f"  {'-'*38}")
effect_rows = []
for col in feature_cols:
    c_vals  = crisis_df[col].dropna()
    nc_vals = non_crisis_df[col].dropna()
    if len(c_vals) < 10 or len(nc_vals) < 10:
        continue
    pooled_std = np.sqrt((c_vals.std()**2 + nc_vals.std()**2) / 2)
    if pooled_std == 0:
        continue
    d = (c_vals.mean() - nc_vals.mean()) / pooled_std
    mag = "Large" if abs(d) >= 0.8 else ("Medium" if abs(d) >= 0.5 else "Small")
    effect_rows.append({"Feature": col, "Family": features[col]["family"],
                        "Window": features[col]["window"],
                        "Cohens_d": round(d, 4), "Magnitude": mag})
    print(f"  {col:<12} {d:>10.4f} {mag:>12}")

effect_df = pd.DataFrame(effect_rows)
effect_df.to_csv(f"{OUT_DIR}/table03_effect_sizes.csv", index=False)

# =============================================================================
# SECTION 4 — REGIME CONTRAST RATIO TABLE
# =============================================================================
print("\n" + "="*70)
print("  SECTION 4: REGIME CONTRAST RATIOS (Crisis Mean / Non-Crisis Mean)")
print("="*70)

contrast_rows = []
print(f"\n  TABLE 4 — Crisis-to-Non-Crisis Mean Ratio")
print(f"  {'Feature':<12} {'Crisis_Mean':>13} {'NC_Mean':>13} {'Ratio':>8}")
print(f"  {'-'*50}")
for col in feature_cols:
    c_mean  = crisis_df[col].dropna().mean()
    nc_mean = non_crisis_df[col].dropna().mean()
    if nc_mean != 0 and not np.isnan(c_mean) and not np.isnan(nc_mean):
        ratio = c_mean / nc_mean
        contrast_rows.append({
            "Feature": col, "Family": features[col]["family"],
            "Window": features[col]["window"],
            "Crisis_Mean": round(c_mean, 8),
            "NC_Mean": round(nc_mean, 8),
            "Ratio": round(ratio, 4),
        })
        print(f"  {col:<12} {c_mean:>13.6f} {nc_mean:>13.6f} {ratio:>8.3f}x")

contrast_df = pd.DataFrame(contrast_rows)
contrast_df.to_csv(f"{OUT_DIR}/table04_regime_contrast_ratios.csv", index=False)

# =============================================================================
# SECTION 5 — FEATURE MATRIX FOR PARTICIPANT 3
# =============================================================================
print("\n" + "="*70)
print("  SECTION 5: EXPORTING FEATURE MATRIX FOR PARTICIPANT 3")
print("="*70)

p3_cols = ["Log_Return", "Rolling_Vol"] + feature_cols + ["Crisis_Indicator"]
feature_matrix = df[p3_cols].copy()
feature_matrix_clean = feature_matrix.dropna()

feature_matrix_clean.to_csv(f"{OUT_DIR}/FTSE100_Feature_Matrix_P3.csv")
print(f"\n  Feature matrix exported: {len(feature_matrix_clean)} rows × {len(p3_cols)} columns")
print(f"  Crisis rows: {feature_matrix_clean['Crisis_Indicator'].sum()}")
print(f"  Non-Crisis rows: {(feature_matrix_clean['Crisis_Indicator']==0).sum()}")
print(f"  Columns: {list(feature_matrix_clean.columns)}")
print(f"  Missing values: {feature_matrix_clean.isnull().sum().sum()}")

# =============================================================================
# SECTION 6 — VISUALISATIONS (5 original charts, all P2-scoped)
# =============================================================================
print("\n" + "="*70)
print("  SECTION 6: GENERATING VISUALISATIONS")
print("="*70)

# ── CHART P2-01 — Multi-Window Rolling Volatility Time Series (all 5 families)
print("\n  Chart P2-01: Multi-Window Rolling Volatility Time Series …")

families = {
    "RSD":  {"label": "Rolling Std Dev",       "color": "#58a6ff"},
    "MAD":  {"label": "Mean Abs Deviation",    "color": "#bc8cff"},
    "VAR":  {"label": "Rolling Variance",      "color": "#d29922"},
    "RNG":  {"label": "Rolling Range",         "color": "#3fb950"},
    "KURT": {"label": "Rolling Kurtosis",      "color": "#ff7b72"},
}

fig, axes = plt.subplots(5, 1, figsize=(18, 22), sharex=True)
fig.suptitle("FTSE 100 — Five Rolling Volatility Families Across Three Windows (20 / 60 / 120 days)\nCrisis Periods Shaded",
             fontsize=13, y=1.001)

for ax, (fam, fmeta) in zip(axes, families.items()):
    for w in windows:
        col = f"{fam}_{w}"
        if col in df.columns:
            ax.plot(df.index, df[col], lw=0.7, alpha=0.85,
                    color=WIN_COLORS[w], label=f"{w}d")
    ax.fill_between(df.index, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else 0,
                    df[f"{fam}_20"].max() * 1.05 if df[f"{fam}_20"].notna().any() else 1,
                    where=(df["Crisis_Indicator"] == 1),
                    color=CRISIS_COLOR, alpha=0.12, label="Crisis")
    ax.set_title(fmeta["label"], fontsize=10, pad=4)
    ax.set_ylabel("Value", fontsize=8)
    ax.legend(fontsize=8, loc="upper left", ncol=4)
    ax.grid(True, alpha=0.35)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_01_multiwindow_timeseries.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_01_multiwindow_timeseries.png")

# ── CHART P2-02 — Box Plots: All 15 Features, Split by Regime ────────────────
print("  Chart P2-02: Box Plots — All 15 features by regime …")

fig, axes = plt.subplots(5, 3, figsize=(18, 22))
fig.suptitle("FTSE 100 — Regime-Disaggregated Box Plots: All 15 Volatility Features\n(Crisis = Red | Non-Crisis = Green)",
             fontsize=13, y=1.002)

for idx, (col, meta) in enumerate(features.items()):
    row_i = idx // 3
    col_i = idx % 3
    ax = axes[row_i][col_i]

    c_data  = crisis_df[col].dropna()
    nc_data = non_crisis_df[col].dropna()

    bp = ax.boxplot([nc_data, c_data],
                    labels=["Non-Crisis", "Crisis"],
                    patch_artist=True,
                    medianprops=dict(color="#24292f", lw=1.8),
                    flierprops=dict(marker=".", ms=1.5, alpha=0.3),
                    whiskerprops=dict(lw=1.0),
                    capprops=dict(lw=1.0))
    bp["boxes"][0].set_facecolor(NORMAL_COLOR + "55")
    bp["boxes"][1].set_facecolor(CRISIS_COLOR + "55")

    ax.set_title(meta["description"], fontsize=9, pad=3)
    ax.tick_params(labelsize=8)
    ax.grid(True, axis="y", alpha=0.35)

    # Annotate median values
    for i, data in enumerate([nc_data, c_data]):
        med = data.median()
        ax.text(i + 1, med, f"{med:.5f}", ha="center", va="bottom",
                fontsize=6.5, color="#24292f")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_02_boxplots_all_features.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_02_boxplots_all_features.png")

# ── CHART P2-03 — KDE Overlays: Crisis vs Non-Crisis per Volatility Family ────
print("  Chart P2-03: KDE Overlays per family …")

fig, axes = plt.subplots(3, 5, figsize=(22, 14))
fig.suptitle("FTSE 100 — Kernel Density Estimates: Crisis vs Non-Crisis\nAcross All 15 Rolling Volatility Features",
             fontsize=13, y=1.003)

for idx, (col, meta) in enumerate(features.items()):
    row_i = idx // 5
    col_i = idx % 5
    ax = axes[row_i][col_i]

    c_data  = crisis_df[col].dropna()
    nc_data = non_crisis_df[col].dropna()

    if len(c_data) > 5 and len(nc_data) > 5:
        kde_c  = stats.gaussian_kde(c_data)
        kde_nc = stats.gaussian_kde(nc_data)
        x_min  = min(c_data.quantile(0.005), nc_data.quantile(0.005))
        x_max  = max(c_data.quantile(0.995), nc_data.quantile(0.995))
        x      = np.linspace(x_min, x_max, 300)
        ax.fill_between(x, kde_nc(x), color=NORMAL_COLOR, alpha=0.35, label="Non-Crisis")
        ax.fill_between(x, kde_c(x),  color=CRISIS_COLOR, alpha=0.35, label="Crisis")
        ax.plot(x, kde_nc(x), color=NORMAL_COLOR, lw=1.2)
        ax.plot(x, kde_c(x),  color=CRISIS_COLOR, lw=1.2)

    # Annotate KS result
    ks_row = test_df[test_df["Feature"] == col]
    if not ks_row.empty:
        sig = ks_row.iloc[0]["Significance"]
        ax.text(0.97, 0.95, f"KS: {sig}", transform=ax.transAxes,
                ha="right", va="top", fontsize=7.5,
                color=CRISIS_COLOR if sig == "***" else "#24292f",
                bbox=dict(boxstyle="round,pad=0.2", fc="#f6f8fa", ec="#d0d7de"))

    ax.set_title(meta["description"], fontsize=8, pad=3)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=6.5, loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_03_kde_overlays.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_03_kde_overlays.png")

# ── CHART P2-04 — Regime Contrast Heatmap (Crisis Mean / Non-Crisis Mean) ─────
print("  Chart P2-04: Regime Contrast Heatmap …")

# Reshape contrast ratios into matrix: Family × Window
ratio_matrix = pd.DataFrame(index=list(families.keys()), columns=windows, dtype=float)
for _, row in contrast_df.iterrows():
    ratio_matrix.loc[row["Family"], row["Window"]] = row["Ratio"]

# Cohen's d matrix
d_matrix = pd.DataFrame(index=list(families.keys()), columns=windows, dtype=float)
for _, row in effect_df.iterrows():
    d_matrix.loc[row["Family"], row["Window"]] = row["Cohens_d"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap 1: Ratio
sns.heatmap(ratio_matrix.astype(float), annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=axes[0], linewidths=0.5, linecolor="#d0d7de",
            cbar_kws={"label": "Ratio (Crisis / Non-Crisis Mean)"},
            annot_kws={"size": 11})
axes[0].set_title("Crisis-to-Non-Crisis Mean Ratio\n(>1 = Higher During Crisis)", fontsize=11)
axes[0].set_xlabel("Rolling Window (days)")
axes[0].set_ylabel("Volatility Family")

# Heatmap 2: Cohen's d
sns.heatmap(d_matrix.astype(float), annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=axes[1], linewidths=0.5, linecolor="#d0d7de",
            cbar_kws={"label": "Cohen's d (Effect Size)"},
            annot_kws={"size": 11})
axes[1].set_title("Effect Size — Cohen's d\n(Crisis vs Non-Crisis)", fontsize=11)
axes[1].set_xlabel("Rolling Window (days)")
axes[1].set_ylabel("Volatility Family")

fig.suptitle("FTSE 100 — Regime Discrimination Strength Across All 15 Features",
             fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_04_regime_contrast_heatmap.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_04_regime_contrast_heatmap.png")

# ── CHART P2-05 — Violin Plots: 5 Families Across Windows ────────────────────
print("  Chart P2-05: Violin Plots — 5 volatility families …")

fig, axes = plt.subplots(1, 5, figsize=(22, 7))
fig.suptitle("FTSE 100 — Volatility Distribution Shape: Crisis vs Non-Crisis\n(Violin Plots | All Five Families, Short-to-Long Window Comparison)",
             fontsize=12, y=1.02)

for ax, (fam, fmeta) in zip(axes, families.items()):
    rows = []
    for w in windows:
        col = f"{fam}_{w}"
        c_data  = crisis_df[col].dropna()
        nc_data = non_crisis_df[col].dropna()
        for v in c_data:
            rows.append({"Value": v, "Regime": "Crisis", "Window": f"{w}d"})
        for v in nc_data:
            rows.append({"Value": v, "Regime": "Non-Crisis", "Window": f"{w}d"})
    vdf = pd.DataFrame(rows)

    sns.violinplot(data=vdf, x="Window", y="Value", hue="Regime",
                   palette={"Crisis": CRISIS_COLOR, "Non-Crisis": NORMAL_COLOR},
                   ax=ax, inner="quartile", linewidth=0.9, split=False,
                   order=["20d", "60d", "120d"])
    ax.set_title(fmeta["label"], fontsize=9)
    ax.set_xlabel("Window")
    ax.set_ylabel("Value" if ax == axes[0] else "")
    ax.tick_params(labelsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    if ax != axes[0]:
        ax.get_legend().remove() if ax.get_legend() else None
    else:
        ax.legend(fontsize=7.5, title_fontsize=7)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_05_violin_all_families.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_05_violin_all_families.png")

# ── CHART P2-06 — Window-Length Sensitivity: RSD across crises episodes ───────
print("  Chart P2-06: Crisis Episode Zoom — RSD Window Sensitivity …")

# Define major crisis episodes
episodes = {
    "Dot-Com Bust\n(2000–2003)":      ("2000-01-01", "2003-12-31"),
    "GFC\n(2007–2009)":               ("2007-06-01", "2009-12-31"),
    "Euro Debt Crisis\n(2010–2012)":  ("2010-01-01", "2012-12-31"),
    "COVID-19\n(2020)":               ("2019-10-01", "2021-06-30"),
}

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle("FTSE 100 — Rolling Standard Deviation (RSD): Window-Length Sensitivity Across Crisis Episodes",
             fontsize=12, y=1.01)
axes_flat = axes.flatten()

for ax, (ep_name, (start, end)) in zip(axes_flat, episodes.items()):
    mask = (df.index >= start) & (df.index <= end)
    ep_df = df[mask]

    for w in windows:
        col = f"RSD_{w}"
        ax.plot(ep_df.index, ep_df[col], color=WIN_COLORS[w], lw=1.0,
                alpha=0.85, label=f"{w}d RSD")

    ax.fill_between(ep_df.index, 0, ep_df["RSD_20"].max() * 1.1,
                    where=(ep_df["Crisis_Indicator"] == 1),
                    color=CRISIS_COLOR, alpha=0.15, label="Crisis")
    ax.set_title(ep_name, fontsize=10)
    ax.set_xlabel("Date"); ax.set_ylabel("Rolling Std Dev")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.35)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_06_crisis_episode_zoom.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_06_crisis_episode_zoom.png")

# ── CHART P2-07 — Kurtosis & Skewness Rolling Time Series ─────────────────────
print("  Chart P2-07: Rolling Kurtosis Time Series …")

fig, axes = plt.subplots(3, 1, figsize=(18, 13), sharex=True)
fig.suptitle("FTSE 100 — Rolling Kurtosis: Tail Risk Dynamics Across Windows\n(Higher Kurtosis = Fatter Tails = More Extreme Events)",
             fontsize=12, y=1.01)

for ax, w in zip(axes, windows):
    col = f"KURT_{w}"
    ax.plot(df.index, df[col], color=WIN_COLORS[w], lw=0.8, label=f"{w}d Kurtosis")
    ax.axhline(3.0, color=WARN_COLOR, lw=1.0, ls="--", alpha=0.7, label="Normal Kurtosis (3)")
    ax.axhline(0.0, color=NEUTRAL_COLOR, lw=0.7, ls=":", alpha=0.5)
    ax.fill_between(df.index, df[col].min(), df[col].max(),
                    where=(df["Crisis_Indicator"] == 1),
                    color=CRISIS_COLOR, alpha=0.12, label="Crisis")
    ax.set_title(f"{w}-Day Rolling Kurtosis", fontsize=10)
    ax.set_ylabel("Excess Kurtosis")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.35)

axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/chart_P2_07_rolling_kurtosis.png", bbox_inches="tight")
plt.close()
print("    Saved chart_P2_07_rolling_kurtosis.png")

# =============================================================================
# SECTION 7 — ADDITIONAL ANNOTATED STATS TABLE FOR PARTICIPANT 4
# =============================================================================
print("\n" + "="*70)
print("  SECTION 7: EXPORTING ANNOTATED STATS TABLE FOR PARTICIPANT 4")
print("="*70)

p4_export = summary_df[summary_df["Regime"] != "All"].copy()
p4_export = p4_export.merge(test_df[["Feature", "KS_Stat", "KS_p", "MW_Stat", "MW_p", "Significance"]],
                            on="Feature", how="left")
p4_export = p4_export.merge(effect_df[["Feature", "Cohens_d", "Magnitude"]], on="Feature", how="left")
p4_export.to_csv(f"{OUT_DIR}/table05_annotated_stats_for_P4.csv", index=False)
print(f"  Exported annotated stats: {len(p4_export)} rows")

# =============================================================================
# SECTION 8 — SUMMARY PRINTOUT
# =============================================================================
print("\n" + "="*70)
print("  PARTICIPANT 2 — ANALYSIS COMPLETE")
print("="*70)
print(f"\n  Output directory: ./{OUT_DIR}/")
print(f"\n  FILES GENERATED:")
print(f"    [CSV]  table01_descriptive_stats_by_regime.csv")
print(f"    [CSV]  table02_hypothesis_tests.csv")
print(f"    [CSV]  table03_effect_sizes.csv")
print(f"    [CSV]  table04_regime_contrast_ratios.csv")
print(f"    [CSV]  table05_annotated_stats_for_P4.csv")
print(f"    [CSV]  FTSE100_Feature_Matrix_P3.csv   ← FOR PARTICIPANT 3")
print(f"    [PNG]  chart_P2_01_multiwindow_timeseries.png")
print(f"    [PNG]  chart_P2_02_boxplots_all_features.png")
print(f"    [PNG]  chart_P2_03_kde_overlays.png")
print(f"    [PNG]  chart_P2_04_regime_contrast_heatmap.png")
print(f"    [PNG]  chart_P2_05_violin_all_families.png")
print(f"    [PNG]  chart_P2_06_crisis_episode_zoom.png")
print(f"    [PNG]  chart_P2_07_rolling_kurtosis.png")

print(f"\n  FEATURE MATRIX SUMMARY (for Participant 3):")
print(f"    Total observations:  {len(feature_matrix_clean)}")
print(f"    Features computed:   {len(feature_cols)}  (5 families × 3 windows)")
print(f"    Classification target: Crisis_Indicator")
print(f"    Class balance:  Crisis={feature_matrix_clean['Crisis_Indicator'].sum()} "
      f"({feature_matrix_clean['Crisis_Indicator'].mean()*100:.1f}%)  |  "
      f"Non-Crisis={(feature_matrix_clean['Crisis_Indicator']==0).sum()}")

print("\n  STATISTICAL TEST SUMMARY:")
sig_features = test_df[test_df["Significance"] == "***"]
print(f"    Features significant at p<0.01 (both KS & MWU): {len(sig_features)} / {len(test_df)}")
for _, r in sig_features.iterrows():
    print(f"      {r['Feature']:<12}  KS_p={r['KS_p']:.2e}  MWU_p={r['MW_p']:.2e}")

print("\n  Done. ✓")

  PARTICIPANT 2 — ROLLING VOLATILITY COMPUTATION & COMPARATIVE ANALYSIS

  Loaded: 6546 trading days  |  Crisis: 1002  |  Non-Crisis: 5544

  SECTION 1: COMPUTING 15-FEATURE ROLLING VOLATILITY MATRIX

  Computed 15 features:
    RSD_20        |  Rolling Std Dev (20d)           |  Non-null: 6527
    MAD_20        |  Rolling MAD (20d)               |  Non-null: 6527
    VAR_20        |  Rolling Variance (20d)          |  Non-null: 6527
    RNG_20        |  Rolling Range (20d)             |  Non-null: 6527
    KURT_20       |  Rolling Kurtosis (20d)          |  Non-null: 6527
    RSD_60        |  Rolling Std Dev (60d)           |  Non-null: 6487
    MAD_60        |  Rolling MAD (60d)               |  Non-null: 6487
    VAR_60        |  Rolling Variance (60d)          |  Non-null: 6487
    RNG_60        |  Rolling Range (60d)             |  Non-null: 6487
    KURT_60       |  Rolling Kurtosis (60d)          |  Non-null: 6487
    RSD_120       |  Rolling Std Dev (120d)          |  Non-null: